# Search Engine With Langchain

In [1]:
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun
from langchain_community.utilities import ArxivAPIWrapper, WikipediaAPIWrapper

C:\Users\ARJU\AppData\Local\Temp\ipykernel_10376\4123191527.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun


In [2]:
# Used the inbuilt tools of wikipedia
api_wrapper_wiki = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=250)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name

'wikipedia'

In [3]:
# Used the inbuilt tools of arxiv
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=250)
arxiv = ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
arxiv.name

'arxiv'

In [4]:
# Custom Toll [RAG Tool]
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [5]:
loader = WebBaseLoader('https://www.kgec.edu.in/')
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitted_docs = text_splitter.split_documents(docs)

embedding = OllamaEmbeddings(model='nomic-embed-text')

vector_store = FAISS.from_documents(documents=splitted_docs, embedding=embedding)
retriever = vector_store.as_retriever()

In [6]:
# Retriever to tool
from langchain_core.tools import create_retriever_tool
retriever_tool = create_retriever_tool(retriever=retriever, name='KGEC', description='Search any information about KGEC')
retriever_tool.name

'KGEC'

In [7]:
tools = [wiki, arxiv, retriever_tool]
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'd:\\Generative Ai\\Udemy\\venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 StructuredTool(name='KGEC', description='Search any information about KGEC', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x0000027DC97DE160>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x0000027DB02D8220>)]

In [8]:
# Create LLM
from langchain_groq import ChatGroq
import os 
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv('GROQ_API_KEY')

llm = ChatGroq(model='openai/gpt-oss-20b', api_key=groq_api_key)

In [19]:
# Prompts Template
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompts = ChatPromptTemplate(
    [
        ('system', "You are a helpfull assistant. Please response accurately of the user's query"),
        MessagesPlaceholder('chat_history'),
        ('human', 'Question: {input}'),
        MessagesPlaceholder("agent_scratchpad")
    ]
)
prompts

ChatPromptTemplate(input_variables=['agent_scratchpad', 'chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')]

In [20]:
# Create agent
from langchain_classic.agents import create_openai_tools_agent
agent = create_openai_tools_agent(llm=llm, tools=tools, prompt=prompts)
agent

RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChun

In [21]:
# Agent Executer
from langchain_classic.agents import AgentExecutor
agent_executer = AgentExecutor(agent=agent, tools=tools, verbose=True)
agent_executer

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| ChatPromptTemplate(input_variables=['agent_scratchpad', 'chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChu

In [23]:
agent_executer.invoke({'input': 'Tell me about kgec', 'chat_history': []})



> Entering new AgentExecutor chain...

Invoking: `KGEC` with `{'query': 'kgec'}`


Kalyani Government Engineering College (KGEC) | Official Portal🇮🇳Har Ghar TirangaWelcome to KGECNIRFNAACIQACIICAnti-RaggingTendersNoticesBackofficeকল্যাণী গভর্নমেন্ট ইঞ্জিনিয়ারিং কলেজKalyani Government Engineering CollegeA Government of West Bengal InstituteAffiliated to MAKAUTAICTE ApprovedGovt. of West Bengal | AICTE ApprovedGoverning Body Chairman: Prof. Kallol Paul, Vice-Chancellor, Kalyani UniversityKGEC WebsiteHome PageAdmissionsAbout Academics Admissions Departments Compliance & Cells Research Placements Alumni NoticesContactKGEC Official PortalKGEC WebsiteHomeActiveAboutAcademicsAdmissionsDepartmentsCompliance & CellsResearchPlacementsAlumniNoticesContactFaculty & Staff BackofficeApply for Admissions 2026Placement Brochure (PDF)Since 1995, KGEC has kept growing on the Kalyani campus.Explore MoreInstitutional Overview31+Years of ExcellenceEstablished in 19952,400+Students EnrolledUG, PG & Docto

{'input': 'Tell me about kgec',
 'chat_history': [],
 'output': '**Kalyani Government Engineering College (KGEC)**  \n*Location:* Kalyani, West Bengal, India  \n*Established:* 1995  \n*Affiliation:* Maulana Abul Kalam Azad University of Technology (MAKAUT)  \n*Approval:* All India Council for Technical Education (AICTE)\n\n---\n\n### Overview\n- **Mission:** Provide high‑quality technical education to undergraduate and postgraduate students.  \n- **Reputation:** One of the leading government engineering institutions in Eastern India, known for strong academics, research, and placement performance.\n\n### Academic Programs\n| Level | Degree | Duration |\n|-------|--------|----------|\n| UG | B.Tech (Computer Science & Engineering, Electronics & Communication, Mechanical, Electrical, Information Technology) | 4 years |\n| PG | M.Tech (various specializations) | 2 years |\n| PG | Master of Computer Applications (MCA) | 3 years |\n| Ph.D. | Engineering & Applied Sciences | 3–5 years |\n\n-

In [24]:
agent_executer.invoke({'input': 'Who is Virat Kohli?', 'chat_history': []})



> Entering new AgentExecutor chain...
Virat Kohli is an Indian international cricketer widely regarded as one of the best batsmen in the modern era. Born on 5 November 1988 in Delhi, he rose through the ranks of youth and domestic cricket to debut for India’s senior team in 2008. Kohli has captained India in all three formats (test, ODI, T20I) and is known for his aggressive batting style, consistency, and leadership. He has amassed numerous records, including being the fastest to 8,000, 9,000, 10,000, and 11,000 runs in One Day Internationals, and he has led India to several major victories, such as the 2017 ICC Champions Trophy and the 2018–19 ICC World Test Championship. Beyond cricket, Kohli is also a prominent brand ambassador and philanthropist.

> Finished chain.


{'input': 'Who is Virat Kohli?',
 'chat_history': [],
 'output': 'Virat Kohli is an Indian international cricketer widely regarded as one of the best batsmen in the modern era. Born on 5\u202fNovember\u202f1988 in Delhi, he rose through the ranks of youth and domestic cricket to debut for India’s senior team in 2008. Kohli has captained India in all three formats (test, ODI, T20I) and is known for his aggressive batting style, consistency, and leadership. He has amassed numerous records, including being the fastest to 8,000, 9,000, 10,000, and 11,000 runs in One Day Internationals, and he has led India to several major victories, such as the 2017 ICC Champions Trophy and the 2018–19 ICC World Test Championship. Beyond cricket, Kohli is also a prominent brand ambassador and philanthropist.'}